# FHIRscan — DIZ-Vergleichsanalyse

Vergleicht die Profiling-Ergebnisse von bis zu **6 Datenintegrationszentren (DIZ)**.

**Aufbau:**
1. **Gesamtübersicht** — Ressourcenzahlen, Vollständigkeit auf einen Blick
2. **Feldvollständigkeit** — Welche Felder sind wie gut befüllt? (interaktive Heatmaps)
3. **Größte Abweichungen** — Wo unterscheiden sich die DIZ am stärksten?
4. **Datenqualität** — Verlinkungsraten zwischen Ressourcentypen
5. **Kardinalität** — Wie viele Ressourcen hat ein Patient je Typ?
6. **Wertverteilungen** — Welche Codes / Werte kommen wie häufig vor?
7. **Profilversionen** — Welche MI-I Profile und Versionen werden genutzt?

> Alle Plotly-Grafiken sind interaktiv: Hover für Details, Legende klicken zum Ein-/Ausblenden.


## ⚙️ Setup

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.3f}'.format)


## ⚙️ Konfiguration

Trage hier die **Output-Ordner** der DIZ ein (die timestamped Unterordner von `Output/`).
Nicht genutzte Einträge können auskommentiert werden.


In [2]:
# ── Konfiguration: Namen und Pfade der DIZ ────────────────────────────────
BASE = Path("..")

DIZ = {
    "MI-I Testdaten":        BASE / "Output"       / "20260520_14_45",
    "Würzburg Testserver":   BASE / "diz_profiles" / "Würzburg_01",
    "Würzburg großer Server":BASE / "diz_profiles" / "Würzburg_02",
    "LMU synth":             BASE / "diz_profiles" / "LMU_synth",
    # "UKA": BASE / "Analysis" / "Augsburg_01",
    # "FAU": BASE / "Analysis" / "Erlangen_01",
}

DIZ_COLORS = px.colors.qualitative.Bold
COLOR_MAP = {name: DIZ_COLORS[i % len(DIZ_COLORS)] for i, name in enumerate(DIZ)}

print(f"Konfigurierte DIZ: {list(DIZ.keys())}")


Konfigurierte DIZ: ['MI-I Testdaten', 'Würzburg Testserver', 'Würzburg großer Server', 'LMU synth']


## 📥 Daten laden

In [3]:
def load_csv_safe(path: Path, diz_name: str) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_csv(path)
    df["diz"] = diz_name
    return df


def load_fields(folder: Path, diz_name: str) -> pd.DataFrame:
    dfs = []
    for csv_file in sorted(folder.glob("*_fields.csv")):
        resource_type = csv_file.stem.replace("_fields", "")
        df = pd.read_csv(csv_file)
        df["diz"] = diz_name
        df["resource_type"] = resource_type
        for col in ["presence_rate", "missing_rate"]:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def parse_top_values(tv_str) -> dict:
    if pd.isna(tv_str) or tv_str == "":
        return {}
    try:
        pairs = json.loads(tv_str)
        return {str(k): int(v) for k, v in pairs}
    except Exception:
        return {}


summary_frames, quality_frames, card_frames, fields_frames = [], [], [], []
CORE_FILES = {
    "summary":     "01_summary.csv",
    "cardinality": "03_cardinality.csv",
    "quality":     "04_data_quality.csv",
}

for diz_name, folder in DIZ.items():
    if not folder.exists():
        print(f"⚠️  Ordner nicht gefunden: {diz_name} → {folder}")
        continue
    summary_frames.append(load_csv_safe(folder / "01_summary.csv", diz_name))
    quality_frames.append(load_csv_safe(folder / "04_data_quality.csv", diz_name))
    card_frames.append(load_csv_safe(folder / "03_cardinality.csv", diz_name))
    fields_frames.append(load_fields(folder, diz_name))

df_summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
df_quality = pd.concat(quality_frames, ignore_index=True) if quality_frames else pd.DataFrame()
df_card    = pd.concat(card_frames,    ignore_index=True) if card_frames    else pd.DataFrame()
df_fields  = pd.concat(fields_frames,  ignore_index=True) if fields_frames  else pd.DataFrame()

LOADED_DIZ = [n for n in DIZ if any(
    (DIZ[n] / f).exists() for f in CORE_FILES.values()
)]
ALL_RT = sorted(df_fields["resource_type"].unique().tolist()) if not df_fields.empty else []

# Filter structural container nodes (pure object/array fields)
_STRUCTURAL = {"object", "array"}
def _is_data_field(type_str) -> bool:
    if pd.isna(type_str) or str(type_str).strip() == "":
        return False
    return bool({t.strip() for t in str(type_str).split(",")} - _STRUCTURAL)

if not df_fields.empty and "detected_python_types" in df_fields.columns:
    df_fields["_is_data_field"] = df_fields["detected_python_types"].apply(_is_data_field)

print(f"Geladen: {len(LOADED_DIZ)} DIZ · {len(ALL_RT)} Ressourcentypen")


Geladen: 4 DIZ · 18 Ressourcentypen


In [4]:
from datetime import datetime
from IPython.display import display, HTML

_now = datetime.now().strftime("%d.%m.%Y %H:%M")

_res_per_diz = (
    df_summary.groupby("diz")["total_resources"].sum()
    if not df_summary.empty else pd.Series(dtype=int)
)
_pat_per_diz = (
    df_summary[df_summary["resource_type"] == "Patient"]
    .set_index("diz")["total_resources"]
    if not df_summary.empty else pd.Series(dtype=int)
)

_rows = "".join(
    f"<tr><td style='padding:5px 12px'>{diz}</td>"
    f"<td style='text-align:right;padding:5px 12px'>{int(_res_per_diz.get(diz, 0)):,}</td>"
    f"<td style='text-align:right;padding:5px 12px'>{int(_pat_per_diz.get(diz, 0)):,}</td></tr>"
    for diz in LOADED_DIZ
)

display(HTML(f'''
<div style="border:1px solid #d0d0d0;border-radius:8px;padding:20px 28px;
            background:#f7f9fc;font-family:sans-serif;max-width:760px">
  <h2 style="margin-top:0;color:#1a1a2e">FHIRscan &mdash; DIZ-Vergleichsreport</h2>
  <p style="color:#666;margin:0 0 14px 0">Generiert am <b>{_now}</b></p>
  <table style="border-collapse:collapse;width:100%;background:#fff;
                border:1px solid #e0e0e0;border-radius:4px">
    <tr style="background:#eef2f7">
      <th style="text-align:left;padding:6px 12px">DIZ</th>
      <th style="text-align:right;padding:6px 12px">Ressourcen gesamt</th>
      <th style="text-align:right;padding:6px 12px">Patienten</th>
    </tr>
    {_rows}
  </table>
  <p style="margin:14px 0 0 0;color:#555;font-size:0.95em">
    <b>{len(LOADED_DIZ)}&thinsp;DIZ</b> &nbsp;&middot;&nbsp;
    <b>{len(ALL_RT)}&thinsp;Ressourcentypen</b> &nbsp;&middot;&nbsp;
    <b>{df_fields["field_path"].nunique() if not df_fields.empty else 0:,}&thinsp;FHIR-Elemente</b>
  </p>
</div>
'''))


DIZ,Ressourcen gesamt,Patienten
MI-I Testdaten,"81,679","12,101"
Würzburg Testserver,"23,957,139","1,766,849"
Würzburg großer Server,"307,749,847","1,559,225"
LMU synth,"31,909,780","4,100,747"


***
## 1. Gesamtübersicht

Ein erster Blick auf die Größe der Datensätze und die mittlere Vollständigkeit je Ressourcentyp.


### 1a. Ressourcenzahlen je Ressourcentyp

Wie viele Ressourcen hat jedes DIZ pro Typ? Balken nebeneinander = verschiedene DIZ.
In der Legende auf einen DIZ-Namen klicken blendet ihn ein oder aus.


In [5]:
# ── Ressourcenzahlen je Ressourcentyp und DIZ ─────────────────────────────
# Tipp: Klick auf DIZ-Namen in der Legende blendet diese ein/aus.
if not df_summary.empty:
    # Detect large gaps that need a broken y-axis
    _vals = np.sort(df_summary["total_resources"].dropna().values.astype(float))
    _gaps = np.diff(_vals) / (_vals[:-1] + 1) if len(_vals) > 1 else np.array([])
    _needs_break = len(_gaps) > 0 and float(np.max(_gaps)) > 2.0

    if _needs_break:
        _bi = int(np.argmax(_gaps))
        _break_lo = _vals[_bi] * 1.15
        _break_hi = _vals[_bi + 1] * 0.85
        _y_max = _vals[-1] * 1.1

        fig = make_subplots(
            rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
            row_heights=[0.2, 0.8],
        )
        for diz_name in LOADED_DIZ:
            sub = df_summary[df_summary["diz"] == diz_name]
            kw = dict(
                name=diz_name, x=sub["resource_type"].tolist(),
                y=sub["total_resources"].tolist(),
                marker_color=COLOR_MAP[diz_name], offsetgroup=diz_name,
                text=[f"{int(v):,}" for v in sub["total_resources"]],
                textposition="outside", textfont_size=9,
            )
            fig.add_trace(go.Bar(**kw, showlegend=False, legendgroup=diz_name), row=1, col=1)
            fig.add_trace(go.Bar(**kw, showlegend=True,  legendgroup=diz_name), row=2, col=1)

        fig.update_yaxes(range=[_break_hi, _y_max], row=1, col=1)
        fig.update_yaxes(range=[0, _break_lo], row=2, col=1, title_text="Anzahl Ressourcen")
        fig.update_xaxes(tickangle=-30)
        fig.update_layout(
            barmode="group", height=550, title="Ressourcenzahlen je Typ und DIZ",
            legend_title="DIZ", bargap=0.15, bargroupgap=0.05,
        )
    else:
        fig = go.Figure()
        for diz_name in LOADED_DIZ:
            sub = df_summary[df_summary["diz"] == diz_name]
            fig.add_trace(go.Bar(
                name=diz_name, x=sub["resource_type"].tolist(),
                y=sub["total_resources"].tolist(),
                marker_color=COLOR_MAP[diz_name],
                text=[f"{int(v):,}" for v in sub["total_resources"]],
                textposition="outside", textfont_size=9,
            ))
        fig.update_layout(
            barmode="group", height=480,
            title="Ressourcenzahlen je Typ und DIZ",
            legend_title="DIZ", xaxis_tickangle=-30, bargroupgap=0.05,
        )

    fig.show()
else:
    print("Keine Summary-Daten geladen.")


### 1b. Ø Presence Rate — Radar-Übersicht

Zeigt auf einen Blick, wie vollständig die Felder je Ressourcentyp befüllt sind.
100 % = alle Felder immer vorhanden. Auf DIZ-Namen in der Legende klicken zum Vergleich.


In [6]:
# ── Ø Presence Rate je Ressourcentyp — interaktives Radar-Chart ───────────
# Zeigt auf einen Blick, welche Ressourcentypen in welchem DIZ gut befüllt sind.
# Hover für genaue Werte, Legende zum Ein-/Ausblenden von DIZ klicken.
if not df_fields.empty and ALL_RT:
    scalar = df_fields[df_fields["_is_data_field"]].copy()
    avg_pr = (
        scalar.groupby(["diz", "resource_type"])["presence_rate"]
        .mean()
        .unstack("resource_type")
        .reindex(columns=ALL_RT)
        .fillna(0)
    )

    fig = go.Figure()
    for diz_name in LOADED_DIZ:
        if diz_name not in avg_pr.index:
            continue
        values = avg_pr.loc[diz_name].tolist()
        theta  = avg_pr.columns.tolist()
        # Close the loop
        fig.add_trace(go.Scatterpolar(
            r=values + [values[0]],
            theta=theta + [theta[0]],
            fill="toself",
            name=diz_name,
            opacity=0.65,
            line_color=COLOR_MAP[diz_name],
            hovertemplate="<b>%{theta}</b><br>Ø Presence Rate: %{r:.1%}<extra>%{fullData.name}</extra>",
        ))

    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True, range=[0, 1],
                tickvals=[0.25, 0.5, 0.75, 1.0],
                ticktext=["25 %", "50 %", "75 %", "100 %"],
                tickfont_size=9,
            )
        ),
        title="Ø Presence Rate je Ressourcentyp und DIZ",
        height=520,
        legend_title="DIZ",
    )
    fig.show()
else:
    print("Keine Felddaten verfügbar.")


***
## 2. Feldvollständigkeit

### 2a. Vollständigkeitsklassen je Ressourcentyp

Felder werden in drei Gruppen eingeteilt:
- **Immer (100 %)** — in jeder Ressource vorhanden
- **Manchmal (1–99 %)** — unregelmäßig befüllt
- **Nie (0 %)** — in keiner Ressource vorhanden

Unterschiedliche Schraffierungen = verschiedene DIZ.


In [7]:
# ── Feldvollständigkeit je Ressourcentyp — gestapelte Balken ─────────────
# Felder werden in drei Klassen eingeteilt: Immer (100%) / Manchmal (1–99%) / Nie (0%).
# Balken nebeneinander = verschiedene DIZ, gleiches Feld-Universum des jeweiligen DIZ.
if not df_fields.empty:
    scalar = df_fields[df_fields["_is_data_field"]].copy()

    def classify_pr(pr):
        if pr == 1.0:  return "Immer (100 %)"
        if pr > 0:     return "Manchmal (1–99 %)"
        return "Nie (0 %)"

    scalar["cls"] = scalar["presence_rate"].apply(classify_pr)
    CATS    = ["Immer (100 %)", "Manchmal (1–99 %)", "Nie (0 %)"]
    CAT_CLR = {"Immer (100 %)": "#2ecc71", "Manchmal (1–99 %)": "#f1c40f", "Nie (0 %)": "#e74c3c"}
    HATCH   = ["", "/", "\\", "x", "-", "|"]

    counts = (
        scalar.groupby(["diz", "resource_type", "cls"])
        .size().unstack("cls", fill_value=0)
        .reindex(columns=CATS, fill_value=0).reset_index()
    )
    counts["total"] = counts[CATS].sum(axis=1)

    fig = go.Figure()
    for cat in CATS:
        for j, diz_name in enumerate(LOADED_DIZ):
            sub = counts[counts["diz"] == diz_name].copy()
            pct = (sub[cat] / sub["total"].replace(0, 1) * 100).round(1).tolist()
            abs_vals = sub[cat].tolist()
            fig.add_trace(go.Bar(
                name=cat,
                x=sub["resource_type"].tolist(),
                y=abs_vals,
                offsetgroup=diz_name,
                legendgroup=cat,
                showlegend=(j == 0),
                marker=dict(
                    color=CAT_CLR[cat],
                    pattern=dict(shape=HATCH[j % len(HATCH)], fgcolor="rgba(0,0,0,0.25)"),
                ),
                customdata=list(zip(pct, [diz_name] * len(sub))),
                hovertemplate=(
                    "<b>%{customdata[1]}</b><br>"
                    f"{cat}: %{{y}} Felder (%{{customdata[0]}} %)"
                    "<extra></extra>"
                ),
            ))
        # Invisible separator between categories for grouped+stacked
        for j, diz_name in enumerate(LOADED_DIZ):
            sub = counts[counts["diz"] == diz_name].copy()
            fig.add_trace(go.Bar(
                name=diz_name, x=sub["resource_type"].tolist(), y=[0] * len(sub),
                offsetgroup=diz_name, legendgroup=diz_name,
                showlegend=(cat == CATS[0]),
                marker_color=COLOR_MAP[diz_name],
                hoverinfo="skip",
            ))

    fig.update_layout(
        barmode="stack",
        height=500,
        title="Feldvollständigkeit je Ressourcentyp — Anteil Felder nach Befüllungsgrad",
        legend_title="Kategorie / DIZ",
        xaxis_tickangle=-30,
        bargroupgap=0.08,
    )
    fig.show()
else:
    print("Keine Felddaten verfügbar.")


### 2b. Presence-Rate-Heatmap je Ressourcentyp

Jede Zeile = ein Feld, jede Spalte = ein DIZ.
- **Grün** = immer vorhanden (100 %)
- **Gelb** = teilweise vorhanden
- **Rot** = selten oder nie vorhanden

Die Prozentzahl steht direkt in der Zelle. Felder sind nach mittlerer Befüllrate sortiert.


In [8]:
# ── Presence-Rate-Heatmap je Ressourcentyp ───────────────────────────────
# Grün = Feld immer befüllt, Gelb = teilweise, Rot = selten/nie.
# Prozentangaben in jeder Zelle. Reihenfolge: häufigste Felder oben.
_hm_l_margin = (
    max(180, min(int(max(len(s) for s in df_fields["field_path"].unique()) * 7), 500))
    if not df_fields.empty else 280
)

def presence_heatmap(resource_type: str, min_presence: float = 0.0):
    sub = df_fields[
        (df_fields["resource_type"] == resource_type) &
        df_fields["_is_data_field"]
    ].copy()
    if sub.empty:
        return

    pivot = (
        sub.pivot_table(index="field_path", columns="diz", values="presence_rate", aggfunc="first")
        .reindex(columns=LOADED_DIZ)
    )
    pivot = pivot[pivot.max(axis=1) > min_presence]
    if pivot.empty:
        return

    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]

    # Percentage text for each cell
    text = [
        [f"{v:.0%}" if pd.notna(v) else "—" for v in row]
        for row in pivot.values
    ]

    height = max(420, len(pivot) * 22 + 120)
    width  = len(LOADED_DIZ) * 100 + _hm_l_margin + 160

    fig = px.imshow(
        pivot,
        color_continuous_scale="RdYlGn",
        zmin=0, zmax=1,
        aspect="auto",
        title=f"Presence Rate — {resource_type}  ({len(pivot)} Felder)",
        labels={"color": "Presence Rate"},
        height=height, width=width,
        text_auto=False,
    )
    # Overlay percentage text
    fig.update_traces(text=text, texttemplate="%{text}", textfont_size=9)
    fig.update_xaxes(side="top")
    fig.update_layout(
        margin=dict(l=_hm_l_margin, r=130, t=80, b=20),
        coloraxis_colorbar=dict(
            title="Rate",
            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
            ticktext=["0 %", "25 %", "50 %", "75 %", "100 %"],
        ),
    )
    fig.show()


if not df_fields.empty:
    for rt in ALL_RT:
        presence_heatmap(rt)
else:
    print("Keine Felddaten verfügbar.")


### 2c. Größte Abweichungen zwischen DIZ

Welche Felder unterscheiden sich am stärksten zwischen den DIZ?
**Ressourcentyp über das Dropdown rechts oben auswählen.**
Hover über einen Balken zeigt die Einzelwerte je DIZ.


In [9]:
# ── Größte Abweichungen zwischen DIZ — interaktiv nach Ressourcentyp ──────
# Zeigt Felder, bei denen die DIZ am stärksten voneinander abweichen.
# Ressourcentyp über das Dropdown rechts oben auswählen.
TOP_N = 20
MIN_PRESENCE = 0.01

if not df_fields.empty and len(LOADED_DIZ) > 1:
    scalar = df_fields[df_fields["_is_data_field"]].copy()

    # Build divergence table
    results = []
    for rt in ALL_RT:
        sub = scalar[scalar["resource_type"] == rt]
        pivot = (
            sub.pivot_table(index="field_path", columns="diz", values="presence_rate", aggfunc="first")
            .reindex(columns=LOADED_DIZ)
        )
        pivot = pivot[pivot.max(axis=1) > MIN_PRESENCE]
        if pivot.empty:
            continue
        pivot["range"] = pivot[LOADED_DIZ].max(axis=1) - pivot[LOADED_DIZ].min(axis=1)
        pivot["mean"]  = pivot[LOADED_DIZ].mean(axis=1)
        pivot["resource_type"] = rt
        results.append(pivot.reset_index())

    if results:
        df_div = pd.concat(results, ignore_index=True).sort_values("range", ascending=False)

        fig = go.Figure()
        buttons = []
        trace_idx = 0

        for i, rt in enumerate(ALL_RT):
            sub = df_div[df_div["resource_type"] == rt].nlargest(TOP_N, "range")
            if sub.empty:
                continue

            # Hover text: show per-DIZ values
            hover_lines = []
            for _, row in sub.iterrows():
                diz_vals = "  |  ".join(
                    f"{d}: {row[d]:.0%}" for d in LOADED_DIZ if d in row and pd.notna(row[d])
                )
                hover_lines.append(f"<b>{row['field_path']}</b><br>{diz_vals}<extra></extra>")

            fig.add_trace(go.Bar(
                x=sub["range"].tolist(),
                y=sub["field_path"].tolist(),
                orientation="h",
                name=rt,
                visible=(i == 0),
                text=[f"{v:.0%}" for v in sub["range"]],
                textposition="outside",
                textfont_size=10,
                marker_color=COLOR_MAP.get(rt, DIZ_COLORS[i % len(DIZ_COLORS)]),
                customdata=sub["mean"].tolist(),
                hovertemplate=hover_lines,
            ))
            trace_idx += 1

        # Dropdown buttons — one per resource type
        n_traces = len(fig.data)
        shown = 0
        for i, rt in enumerate(ALL_RT):
            sub = df_div[df_div["resource_type"] == rt]
            if sub.empty:
                continue
            vis = [False] * n_traces
            vis[shown] = True
            buttons.append(dict(
                label=rt,
                method="update",
                args=[
                    {"visible": vis},
                    {"title": f"Top {TOP_N} Felder mit größter Abweichung — {rt}",
                     "yaxis.autorange": True},
                ],
            ))
            shown += 1

        fig.update_layout(
            updatemenus=[dict(
                active=0,
                buttons=buttons,
                direction="down",
                showactive=True,
                x=1.02, xanchor="left",
                y=1.0, yanchor="top",
            )],
            title=f"Top {TOP_N} Felder mit größter Abweichung — {ALL_RT[0] if ALL_RT else ''}",
            height=max(500, TOP_N * 28 + 120),
            xaxis=dict(title="Max − Min Presence Rate", tickformat=".0%", range=[0, 1.1]),
            yaxis=dict(autorange="reversed"),
            margin=dict(r=180),
        )
        fig.show()
else:
    print("Mindestens 2 DIZ für Abweichungsanalyse erforderlich.")


***
## 3. Datenqualität — Verlinkungsraten

Anteil der Ressourcen mit gültiger `subject.reference` auf Patient bzw. Encounter.
Erwartungswert: nahe 100 % für klinische Ressourcen (Observation, Condition, …).


In [10]:
# ── Linkage-Raten: Wie gut sind Ressourcen mit Patient/Encounter verknüpft? ──
# Hohe Rate (grün) = Ressourcen haben gültige Referenz.
# Ressourcentypen ohne Patient-Referenz per Design (Patient, Location, …) werden
# in einer separaten Tabelle unten aufgelistet.
if not df_quality.empty:
    NO_PATIENT_LINK = {"Patient", "Location", "Medication",
                        "Organization", "Practitioner", "Substance"}

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Patient Linkage Rate", "Encounter Linkage Rate"],
        horizontal_spacing=0.12,
    )

    for diz_name in LOADED_DIZ:
        sub = df_quality[df_quality["diz"] == diz_name]
        common = dict(
            name=diz_name, x=sub["resource_type"].tolist(),
            marker_color=COLOR_MAP[diz_name],
        )
        pat_text = [f"{v:.0%}" if pd.notna(v) else "" for v in sub["patient_linkage_rate"]]
        enc_text = [f"{v:.0%}" if pd.notna(v) else "" for v in sub["encounter_linkage_rate"]]

        fig.add_trace(go.Bar(
            **common, y=sub["patient_linkage_rate"].tolist(),
            text=pat_text, textposition="outside", textfont_size=9,
            showlegend=True,
        ), row=1, col=1)

        fig.add_trace(go.Bar(
            **common, y=sub["encounter_linkage_rate"].tolist(),
            text=enc_text, textposition="outside", textfont_size=9,
            showlegend=False,
        ), row=1, col=2)

    fig.update_yaxes(range=[0, 1.15], tickformat=".0%")
    fig.update_layout(
        barmode="group", height=450,
        title="Datenqualität: Verlinkungsraten je Ressourcentyp und DIZ",
        legend_title="DIZ", xaxis_tickangle=-30, xaxis2_tickangle=-30,
    )
    fig.show()

    # Summary of issues
    low = df_quality[
        (df_quality["patient_linkage_rate"] < 0.8) &
        (~df_quality["resource_type"].isin(NO_PATIENT_LINK))
    ][["diz", "resource_type", "patient_linkage_rate", "total_resources"]]

    if not low.empty:
        print("⚠️  Niedrige Patient-Linkage-Rate (< 80 %):")
        display(low.sort_values("patient_linkage_rate").reset_index(drop=True))
    else:
        print("✅ Alle Ressourcentypen haben ausreichende Patient-Linkage-Rate.")
else:
    print("Keine Qualitätsdaten geladen.")


✅ Alle Ressourcentypen haben ausreichende Patient-Linkage-Rate.


***
## 4. Kardinalität — Ressourcen je Patient

Wie viele Ressourcen eines bestimmten Typs hat ein durchschnittlicher Patient?
**Ressourcentyp über das Dropdown rechts oben auswählen.**
- Linkes Panel: Mittlere Anzahl (Fehlerbalken = Standardabweichung)
- Rechtes Panel: Anteil Patienten mit mindestens einer Ressource dieses Typs


In [11]:
# ── Kardinalität: Wie viele Ressourcen hat ein Patient je Typ? ────────────
# Fehlerbalken = Standardabweichung. Rechtes Panel: Anteil Patienten mit ≥ 1 Ressource.
# Ressourcentyp über das Dropdown rechts oben filtern.
if not df_card.empty:
    for anchor in df_card["anchor_type"].unique():
        sub = df_card[df_card["anchor_type"] == anchor].copy()
        anchor_label = "Patient" if "patient" in anchor else "Encounter"
        resource_types = sorted(sub["resource_type"].unique())

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=[
                f"Mittlere Anzahl je {anchor_label}",
                f"Coverage Rate (≥ 1 Ressource)",
            ],
            horizontal_spacing=0.1,
        )

        # Build one set of traces per resource type so we can add a dropdown
        buttons = []
        trace_start = 0

        for i, rt in enumerate(resource_types):
            rt_data = sub[sub["resource_type"] == rt]
            for diz_name in LOADED_DIZ:
                d = rt_data[rt_data["diz"] == diz_name]
                if d.empty:
                    mean_val, std_val, cov_val = [0], [0], [0]
                else:
                    mean_val = d["mean"].fillna(0).tolist()
                    std_val  = d["std_dev"].fillna(0).tolist() if "std_dev" in d else [0]
                    cov_val  = d["coverage_rate"].fillna(0).tolist()

                show = (i == 0)
                mean_text = [f"{v:.1f}" for v in mean_val]
                cov_text  = [f"{v:.0%}" for v in cov_val]

                fig.add_trace(go.Bar(
                    name=diz_name, x=[rt], y=mean_val,
                    error_y=dict(type="data", array=std_val),
                    marker_color=COLOR_MAP[diz_name],
                    showlegend=(i == 0),
                    visible=show,
                    text=mean_text, textposition="outside", textfont_size=9,
                    legendgroup=diz_name,
                ), row=1, col=1)

                fig.add_trace(go.Bar(
                    name=diz_name, x=[rt], y=cov_val,
                    marker_color=COLOR_MAP[diz_name],
                    showlegend=False,
                    visible=show,
                    text=cov_text, textposition="outside", textfont_size=9,
                    legendgroup=diz_name,
                ), row=1, col=2)

        # Build dropdown
        n_diz = len(LOADED_DIZ)
        total_traces = len(fig.data)
        for i, rt in enumerate(resource_types):
            vis = [False] * total_traces
            # Each RT has n_diz * 2 traces (mean + coverage for each DIZ)
            start = i * n_diz * 2
            for t in range(start, start + n_diz * 2):
                if t < total_traces:
                    vis[t] = True
            buttons.append(dict(
                label=rt,
                method="update",
                args=[{"visible": vis}, {"title": f"Kardinalität ({anchor_label}): {rt}"}],
            ))

        fig.update_layout(
            updatemenus=[dict(
                active=0,
                buttons=buttons,
                direction="down",
                showactive=True,
                x=1.02, xanchor="left",
                y=1.0, yanchor="top",
            )],
            barmode="group", height=450,
            title=f"Kardinalität ({anchor_label}): {resource_types[0] if resource_types else ''}",
            legend_title="DIZ",
            margin=dict(r=200),
        )
        fig.update_yaxes(tickformat=".0%", row=1, col=2)
        fig.update_yaxes(range=[0, fig.data[0].y[0] * 1.25 + 0.5] if fig.data else None, row=1, col=1)
        fig.show()

    # Detail table
    print("\nDetailtabelle Kardinalität (per_patient):")
    detail_cols = ["diz","resource_type","total_resources","min","mean","median","max","p90","p99","coverage_rate"]
    avail = [c for c in detail_cols if c in df_card.columns]
    display(
        df_card[df_card["anchor_type"] == "per_patient"][avail]
        .sort_values(["resource_type", "diz"])
        .reset_index(drop=True)
    )
else:
    print("Keine Kardinalitätsdaten geladen.")



Detailtabelle Kardinalität (per_patient):


,diz,resource_type,total_resources,min,mean,median,max,p90,p99,coverage_rate
0,Würzburg großer Server,AdverseEvent,64364,1,2.580,2.000,26,5,11,0.016
1,Würzburg großer Server,CarePlan,96105,1,2.940,2.000,37,6,16,0.021
2,Würzburg großer Server,ClinicalImpression,46000,1,1.000,1.000,2,1,1,0.029
3,Würzburg großer Server,Composition,2151484,1,6.340,3.000,221,16,47,0.218
4,LMU synth,Condition,3999963,1,1.000,1.000,1,1,1,0.975
5,MI-I Testdaten,Condition,12342,1,1.020,1.000,10,1,1,1.000
6,Würzburg Testserver,Condition,3604536,1,48.500,27.000,1716,115,316,0.042
7,Würzburg großer Server,Condition,9784085,1,15.010,5.000,1431,38,143,0.418
8,MI-I Testdaten,Consent,9687,1,1.000,1.000,1,1,1,0.800
9,Würzburg Testserver,Consent,276,1,1.000,1.000,1,1,1,0.000


***
## 5. Wertverteilungen

Welche Codes / Werte kommen in bestimmten Feldern wie häufig vor?
Passe `VALUE_FIELDS` unten an die tatsächlich vorhandenen Feldpfade in deinen Daten an.


In [12]:
# ── Welche kodierten Felder anzeigen? ─────────────────────────────────────
# Passe diese Liste an deine tatsächlichen Feldpfade an.
VALUE_FIELDS = [
    "Observation.code.coding[].code",
    "Condition.code.coding[].code",
    "Encounter.class.code",
    "Observation.status",
    "Condition.clinicalStatus.coding[].code",
    "Procedure.status",
]
TOP_N_VALUES = 15


In [13]:
# ── Auto-Erkennung: Felder mit interessanter Wertverteilung ───────────────
# Diese Liste zeigt Kandidaten für VALUE_FIELDS (falls Felder oben fehlen).
if not df_fields.empty:
    candidates = df_fields[
        (df_fields.get("detected_fhir_type", pd.Series(dtype=str)).isin(
            ["code", "CodeableConcept", "Coding", "uri"])) &
        (df_fields["presence_rate"] > 0.5) &
        df_fields["_is_data_field"] &
        (df_fields["unique_count"] > 1) &
        (df_fields["unique_count"] < 500)
    ].copy() if "detected_fhir_type" in df_fields.columns else pd.DataFrame()

    if not candidates.empty:
        print("Vorschläge für VALUE_FIELDS (gut befüllte kodierte Felder):")
        display(
            candidates.groupby(["resource_type", "field_path"])["unique_count"]
            .mean().sort_values(ascending=False).head(20)
            .reset_index().rename(columns={"unique_count": "Ø unique_count"})
        )


Vorschläge für VALUE_FIELDS (gut befüllte kodierte Felder):


,resource_type,field_path,Ø unique_count
0,Medication,Medication.code.coding[].code,361.000
1,Consent,Consent.identifier[].value,276.000
2,Consent,Consent.patient.reference,276.000
3,Observation,Observation.code.text,247.000
4,Medication,Medication.identifier[].value,242.000
5,MedicationStatement,MedicationStatement.medicationReference.reference,217.000
6,Medication,Medication.code.text,210.000
7,Observation,Observation.code.coding[].code,177.750
8,Condition,Condition.code.coding[].code,145.000
9,Patient,Patient.address[].country,128.000


In [14]:
# ── Wertverteilung für ausgewählte Felder ─────────────────────────────────
def plot_value_distribution(field_path: str):
    sub = df_fields[df_fields["field_path"] == field_path]
    if sub.empty:
        print(f"Feld '{field_path}' nicht gefunden.")
        return

    rows = []
    for _, row in sub.iterrows():
        tv = parse_top_values(row.get("top_values", ""))
        total = row.get("value_count", 1) or 1
        for value, count in list(tv.items())[:TOP_N_VALUES]:
            rows.append({"diz": row["diz"], "value": value, "count": count, "total": total})

    if not rows:
        print(f"Keine top_values für '{field_path}'.")
        return

    df_tv = pd.DataFrame(rows)
    df_tv["rate"] = df_tv["count"] / df_tv["total"]

    top_vals = (
        df_tv.groupby("value")["count"].sum()
        .sort_values(ascending=False).head(TOP_N_VALUES).index.tolist()
    )
    df_tv = df_tv[df_tv["value"].isin(top_vals)]

    fig = px.bar(
        df_tv, x="value", y="rate", color="diz",
        barmode="group", color_discrete_map=COLOR_MAP,
        title=f"Wertverteilung: {field_path}",
        labels={"rate": "Anteil Feldwerte", "value": "Wert", "diz": "DIZ"},
        text_auto=".1%", height=420,
    )
    fig.update_traces(textposition="outside", textfont_size=9)
    fig.update_yaxes(tickformat=".0%", range=[0, df_tv["rate"].max() * 1.15])
    fig.update_layout(legend_title="DIZ", xaxis_tickangle=-30)
    fig.show()


for field in VALUE_FIELDS:
    plot_value_distribution(field)


***
## 6. MI-I Profilversionen

Welche MI-I Profile und Versionen sind auf den DIZ-Servern hinterlegt?
Orange markierte Zellen = Versionsunterschied zwischen DIZ.


In [15]:
# ── MI-I Profil-Extraktion ─────────────────────────────────────────────────
PROFILE_DOMAIN_MAP = [
    ("medizininformatik-initiative.de/fhir/core", "MII Core"),
    ("medizininformatik-initiative.de/fhir/ext",  "MII Ext"),
    ("medizininformatik-initiative.de/",          "MI-I sonstige"),
    ("gematik.de/fhir/isik",                      "ISiK (Gematik)"),
    ("kbv.de/",                                    "KBV"),
    ("hl7.org/",                                   "HL7"),
]

def _profile_domain(url: str) -> str:
    for pattern, label in PROFILE_DOMAIN_MAP:
        if pattern in url:
            return label
    return "Sonstige"


def extract_profiles(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    mask = df["field_path"].str.endswith(".meta.profile[]", na=False)
    for _, frow in df[mask].iterrows():
        tv = parse_top_values(frow.get("top_values", ""))
        total = int(frow.get("total_resources", 0) or 1)
        for url, cnt in tv.items():
            base_url, version = (url.rsplit("|", 1) if "|" in url else (url, "(ohne Version)"))
            profile_name = base_url.rstrip("/").split("/")[-1]
            rows.append(dict(
                diz=frow["diz"], resource_type=frow["resource_type"],
                profile_url=url, base_url=base_url, profile_name=profile_name,
                version=version, count=int(cnt), total_resources=total,
                share=int(cnt) / total, domain=_profile_domain(url),
            ))
    return pd.DataFrame(rows) if rows else pd.DataFrame()


df_profiles = extract_profiles(df_fields) if not df_fields.empty else pd.DataFrame()
print(f"Profil-Datensätze: {len(df_profiles)}")


Profil-Datensätze: 83


In [16]:
# ── Dominante Profilversion je DIZ ────────────────────────────────────────
# Zeigt welche MI-I Profilversion jedes DIZ für jeden Ressourcentyp nutzt.
if not df_profiles.empty:
    dom = (
        df_profiles.sort_values("count", ascending=False)
        .groupby(["diz", "resource_type", "base_url", "profile_name", "domain"])
        .agg(version=("version", "first"), count=("count", "sum"))
        .reset_index()
    )
    dom["cell"] = dom["version"] + "  (" + dom["count"].apply(lambda x: f"{x:,}") + ")"

    pivot_12a = (
        dom.pivot_table(
            index=["resource_type", "profile_name", "domain"],
            columns="diz", values="cell", aggfunc="first",
        )
        .reindex(columns=LOADED_DIZ)
        .fillna("—").reset_index()
    )
    pivot_12a.columns.name = None

    def _grey_missing(col):
        return ["color: #bbb; font-style: italic" if v == "—" else "" for v in col]

    display(
        pivot_12a.style
        .set_caption("Dominante Profilversion je DIZ  ·  Format: Version  (Ressourcenanzahl)")
        .apply(_grey_missing, subset=LOADED_DIZ)
    )
else:
    print("Keine Profildaten verfügbar.")


,resource_type,profile_name,domain,MI-I Testdaten,Würzburg Testserver,Würzburg großer Server,LMU synth
0,AdverseEvent,mii-pr-onko-nebenwirkung-adverse-event,MII Ext,—,—,"2026.0.0 (64,364)",—
1,CarePlan,mii-pr-onko-tumorkonferenz,MII Ext,—,—,"2026.0.0 (96,105)",—
2,ClinicalImpression,mii-pr-seltene-clinical-impression,MII Ext,—,—,"2026.0.0 (46,000)",—
3,Composition,mii-pr-bildgebung-semistrukt-befundbericht,MII Ext,—,—,"2026.0.0 (2,151,484)",—
4,Condition,Diagnose,MII Core,"(ohne Version) (12,342)","2025.0.0 (3,604,536)","2024.0.0 (9,659,519)","2025.0.0 (3,999,963)"
5,Condition,mii-pr-onko-diagnose-primaertumor,MII Ext,—,—,"2026.0.0 (53,412)",—
6,Condition,mii-pr-onko-fruehere-tumorerkrankung,MII Ext,—,—,"2026.0.0 (6,862)",—
7,Condition,mii-pr-seltene-clinical-diagnosis,MII Ext,—,—,"2026.0.0 (61,009)",—
8,Condition,mii-pr-seltene-symptom-condition,MII Ext,—,—,"2026.0.0 (3,283)",—
9,Consent,Consent,Sonstige,"(ohne Version) (9,687)",—,—,—


In [17]:
# ── Versionsabweichungen zwischen DIZ ─────────────────────────────────────
if not df_profiles.empty and len(LOADED_DIZ) > 1:
    dom_ver = (
        df_profiles.sort_values("count", ascending=False)
        .groupby(["diz", "resource_type", "base_url", "profile_name"])["version"]
        .first().reset_index()
    )
    ver_pivot = (
        dom_ver.pivot_table(
            index=["resource_type", "base_url", "profile_name"],
            columns="diz", values="version", aggfunc="first",
        )
        .reindex(columns=LOADED_DIZ).reset_index()
    )
    diz_cols = [c for c in LOADED_DIZ if c in ver_pivot.columns]
    has_2 = ver_pivot[diz_cols].notna().sum(axis=1) >= 2

    def _differs(row):
        vals = [row[c] for c in diz_cols if pd.notna(row[c])]
        return len(set(vals)) > 1

    divergent = ver_pivot[has_2 & ver_pivot.apply(_differs, axis=1)].copy().reset_index(drop=True)

    if divergent.empty:
        print("✅ Keine Versionsabweichungen — alle DIZ nutzen die gleiche Profilversion.")
    else:
        print(f"⚠️  {len(divergent)} Profil(e) mit unterschiedlichen Versionen zwischen DIZ:\n")

        def _highlight(row):
            styles = [""] * len(row)
            for i, col in enumerate(row.index):
                if col not in diz_cols:
                    continue
                if pd.isna(row[col]):
                    styles[i] = "color: #bbb; font-style: italic"
                    continue
                present = [row[c] for c in diz_cols if pd.notna(row[c])]
                if len(set(present)) > 1:
                    styles[i] = "background-color: #ffe0b2; font-weight: bold"
            return styles

        display(
            divergent.fillna("—").style
            .set_caption("Profile mit Versionsunterschieden (orange = Abweichung)")
            .apply(_highlight, axis=1)
        )
else:
    print("Mindestens 2 DIZ erforderlich für Versionsvergleich.")


⚠️  9 Profil(e) mit unterschiedlichen Versionen zwischen DIZ:



diz,resource_type,base_url,profile_name,MI-I Testdaten,Würzburg Testserver,Würzburg großer Server,LMU synth
0,Condition,https://www.medizininformatik-initiative.de/fhir/core/modul-diagnose/StructureDefinition/Diagnose,Diagnose,(ohne Version),2025.0.0,2024.0.0,2025.0.0
1,Consent,https://www.medizininformatik-initiative.de/fhir/modul-consent/StructureDefinition/mii-pr-consent-einwilligung,mii-pr-consent-einwilligung,—,2025.0.4,(ohne Version),—
2,Encounter,https://www.medizininformatik-initiative.de/fhir/core/modul-fall/StructureDefinition/KontaktGesundheitseinrichtung,KontaktGesundheitseinrichtung,(ohne Version),2025.0.0,2025.0.0,2025.0.0
3,Medication,https://www.medizininformatik-initiative.de/fhir/core/modul-medikation/StructureDefinition/Medication,Medication,(ohne Version),—,2026.0.0,—
4,MedicationAdministration,https://www.medizininformatik-initiative.de/fhir/core/modul-medikation/StructureDefinition/MedicationAdministration,MedicationAdministration,(ohne Version),—,2026.0.0,2025.0.0
5,MedicationStatement,https://www.medizininformatik-initiative.de/fhir/core/modul-medikation/StructureDefinition/MedicationStatement,MedicationStatement,(ohne Version),—,2026.0.0,2025.0.0
6,Observation,https://www.medizininformatik-initiative.de/fhir/core/modul-labor/StructureDefinition/ObservationLab,ObservationLab,(ohne Version),2025.0.2,(ohne Version),—
7,Patient,https://www.medizininformatik-initiative.de/fhir/core/modul-person/StructureDefinition/PatientPseudonymisiert,PatientPseudonymisiert,—,2025.0.1,(ohne Version),2025.0.0
8,Procedure,https://www.medizininformatik-initiative.de/fhir/core/modul-prozedur/StructureDefinition/Procedure,Procedure,(ohne Version),2025.0.0,(ohne Version),2025.0.0


***
## 7. Export

Speichert die wichtigsten Tabellen als CSV-Dateien im `results/`-Ordner.


In [18]:
# ── Ergebnisse exportieren ─────────────────────────────────────────────────
from datetime import datetime

EXPORT_PATH = Path("../results") / datetime.now().strftime("%Y%m%d_%H_%M")
EXPORT_PATH.mkdir(parents=True, exist_ok=True)
exported = []

if not df_fields.empty:
    scalar_all = df_fields[df_fields["_is_data_field"]]
    piv = (
        scalar_all.pivot_table(
            index=["resource_type", "field_path"], columns="diz",
            values="presence_rate", aggfunc="first",
        )
        .reindex(columns=LOADED_DIZ).reset_index()
    )
    piv.columns.name = None
    out = EXPORT_PATH / "presence_rate_all_fields.csv"
    piv.to_csv(out, index=False)
    exported.append(out)

if 'df_div' in dir() and not df_div.empty:
    out = EXPORT_PATH / "field_divergence.csv"
    df_div.to_csv(out, index=False)
    exported.append(out)

if not df_quality.empty:
    out = EXPORT_PATH / "data_quality_comparison.csv"
    df_quality.to_csv(out, index=False)
    exported.append(out)

if not df_card.empty:
    out = EXPORT_PATH / "cardinality_comparison.csv"
    df_card.to_csv(out, index=False)
    exported.append(out)

if not df_profiles.empty:
    out = EXPORT_PATH / "mii_profile_versions.csv"
    df_profiles.to_csv(out, index=False)
    exported.append(out)

print(f"Export nach: {EXPORT_PATH}")
for p in exported:
    print(f"  ✅ {p.name}")


Export nach: ../results/20260714_15_28
  ✅ presence_rate_all_fields.csv
  ✅ field_divergence.csv
  ✅ data_quality_comparison.csv
  ✅ cardinality_comparison.csv
  ✅ mii_profile_versions.csv
